# 14 CFM扩增化合物冻结与无泄漏折分配

**目的**：基于13号CFM-ID Pilot的实测运行结果，冻结可批量生成的结构、结构标准化规则、
ChemVista Novel排除规则、外折允许表和CFM任务清单。本Notebook只生成计划，不运行大规模CFM。

**与真实谱图的比较必须做，但分两层完成**：

1. 14号读取13号已有的454组同结构、同模式、同加合物、近碰撞能量配对，确认域差距随能量变化，
   并冻结后续实测对照配对表；
2. 15号在批量CFM完成后正式计算modified cosine、谱图熵、峰召回率、强度相关性、嵌入距离和
   real-vs-synthetic域分类器。最终由15号决定CFM应作为预训练、低权重增强还是同等训练样本。

**不允许在14号做的事情**：不读取ChemVista谱图，不评价ChemVista P/B/T性能，不解锁Novel/Transfer，
不根据ChemVista结果调整相似度阈值，不运行Transformer。

## 冻结规则（运行前请确认）

- Morgan指纹：radius=2、2048 bit、不使用手性。
- ChemVista Novel近邻排除：最大Tanimoto `>= 0.80`。
- 新增synthetic-only化合物对外折测试集的结构邻近阻断：Tanimoto `>= 0.40`；该阈值对应现有
  Butina距离阈值0.60。
- CFM结构范围：单一组分、3–90个重原子、50–1200 Da，元素限于C/H/N/O/F/P/S/Cl/Br/I。
- 带电单一组分仅在RDKit去电荷后二维InChIKey保持不变且无残余形式电荷时允许；原始结构和转换结构同时保存。
- Development同一化合物CFM谱在其自身测试折中禁止进入训练；synthetic-only化合物按每个测试折的
  最大结构相似度生成允许表。
- 每个化合物的正/负模式和10/20/40 eV谱必须继承同一折权限。

若更改任何阈值，应建立新版本目录，不能覆盖本版本后再把结果称为预先冻结。

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 240)
sns.set_theme(style='whitegrid', context='notebook')

# -------------------- 冻结配置 --------------------
SEED = 20260905
NOVEL_NEIGHBOR_TANIMOTO = 0.80
OUTER_FOLD_BLOCK_TANIMOTO = 0.40
MORGAN_RADIUS = 2
MORGAN_BITS = 2048
MORGAN_USE_CHIRALITY = False
MIN_EXACT_MW = 50.0
MAX_EXACT_MW = 1200.0
MIN_HEAVY_ATOMS = 3
MAX_HEAVY_ATOMS = 90
ALLOWED_ELEMENTS = {'C', 'H', 'N', 'O', 'F', 'P', 'S', 'Cl', 'Br', 'I'}
COLLISION_ENERGIES_EV = [10, 20, 40]
MODES = {'positive': '[M+H]+', 'negative': '[M-H]-'}
ALLOW_OVERWRITE_FROZEN_PLAN = False
# ------------------------------------------------

cwd = Path.cwd().resolve()
roots = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((path for path in roots if (path / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请手动设置 ROOT。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
SPLIT_DIR = BASE / 'split_v1'
AUDIT_DIR = BASE / 'spectrum_audit_v1'
PILOT_DIR = BASE / 'cfm_v1' / 'pilot_v1'
OUTPUT_DIR = BASE / 'cfm_v1' / 'frozen_generation_plan_v1'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_MANIFEST_PATH = OUTPUT_DIR / 'cfm_generation_plan_manifest.json'
if FROZEN_MANIFEST_PATH.exists() and not ALLOW_OVERWRITE_FROZEN_PLAN:
    raise RuntimeError(
        f'检测到已冻结的14号计划：{FROZEN_MANIFEST_PATH}。'
        '若只是查看结果，不要重跑；确需重建时建立新版本目录，或明确将 '
        'ALLOW_OVERWRITE_FROZEN_PLAN 改为 True。'
    )

PATHS = {
    'compound_split': SPLIT_DIR / 'compound_split_manifest.csv',
    'development': SPLIT_DIR / 'development_compounds.csv',
    'public_manifest': AUDIT_DIR / 'public_relevant_spectrum_manifest.csv.gz',
    'strict_targets': ROOT / '性质数据' / 'Cleaned_datasets' / '01_integrated_properties' / '07b_PBT_strict_MSMS_modeling_candidates.csv',
    'pilot_manifest': PILOT_DIR / 'cfm_pilot_run_manifest.json',
    'pilot_qc': PILOT_DIR / 'cfm_pilot_qc_checks.csv',
    'pilot_status': PILOT_DIR / 'cfm_execution_status.csv',
    'pilot_sim_to_real': PILOT_DIR / 'cfm_sim_to_real_metrics.csv',
}
missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少14号输入文件：\n' + '\n'.join(missing)

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Python:', sys.executable)
print('RDKit:', rdBase.rdkitVersion)

## 1. 验收13号Pilot

13号有298/300个命令成功，两个失败来自同一个预带双负电荷结构。14号不把它伪装成300/300，
而是按预先给出的成功率门槛和完整失败记录判定为 `PASS_WITH_DOCUMENTED_IONIZATION_FAILURE`。

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

pilot_manifest = read_json(PATHS['pilot_manifest'])
pilot_qc = pd.read_csv(PATHS['pilot_qc'])
pilot_status = pd.read_csv(PATHS['pilot_status'])
pilot_sim_to_real = pd.read_csv(PATHS['pilot_sim_to_real'])

pilot_complete = pilot_status['status'].isin(['success', 'cached_complete'])
overall_success = float(pilot_complete.mean())
success_by_mode = pilot_status.assign(success=pilot_complete).groupby('mode')['success'].mean()
n_success = int(pilot_complete.sum())
n_failed = int((~pilot_complete).sum())
parsed_expected_from_success = n_success * len(COLLISION_ENERGIES_EV)

PILOT_ACCEPTED = bool(
    overall_success >= 0.95
    and success_by_mode.get('positive', 0.0) >= 0.90
    and success_by_mode.get('negative', 0.0) >= 0.90
    and int(pilot_manifest['counts']['simulated_spectra_parsed']) == parsed_expected_from_success
    and int(pilot_manifest['counts']['sim_to_real_pairs']) > 0
    and pilot_manifest.get('external_test_unlocked') is False
    and pilot_manifest['cfm']['docker_image_digest'] != 'unavailable'
)
assert PILOT_ACCEPTED, '13号Pilot未达到进入14号的最低门槛。'
PILOT_DECISION = 'PASS_WITH_DOCUMENTED_IONIZATION_FAILURE' if n_failed else 'PASS_COMPLETE'

pilot_acceptance = pd.DataFrame([
    {'item': 'overall command success', 'observed': overall_success, 'criterion': '>=0.95', 'passed': overall_success >= 0.95},
    {'item': 'positive success', 'observed': success_by_mode.get('positive', np.nan), 'criterion': '>=0.90', 'passed': success_by_mode.get('positive', 0) >= 0.90},
    {'item': 'negative success', 'observed': success_by_mode.get('negative', np.nan), 'criterion': '>=0.90', 'passed': success_by_mode.get('negative', 0) >= 0.90},
    {'item': 'parsed spectra from successful jobs', 'observed': pilot_manifest['counts']['simulated_spectra_parsed'], 'criterion': parsed_expected_from_success, 'passed': pilot_manifest['counts']['simulated_spectra_parsed'] == parsed_expected_from_success},
    {'item': 'sim-to-real pairs', 'observed': len(pilot_sim_to_real), 'criterion': '>0', 'passed': len(pilot_sim_to_real) > 0},
])
pilot_acceptance.to_csv(OUTPUT_DIR / '13_pilot_acceptance_for_14.csv', index=False, encoding='utf-8-sig')
display(pilot_acceptance)
print('PILOT_DECISION:', PILOT_DECISION)

## 2. 13号模拟谱—实测谱差距

这里保存13号结果作为14号冻结依据，不用ChemVista。重点不是要求CFM与真实谱完全相同，而是确认域差异是否系统性依赖
离子模式和碰撞能量。若40 eV差距明显，15号必须分能量审计，Transformer训练也必须保留能量与synthetic域标记。

In [ ]:
pilot_gap_summary = (
    pilot_sim_to_real.groupby(['mode', 'simulated_ce_ev'])
    .agg(
        n_pairs=('cosine', 'size'),
        cosine_mean=('cosine', 'mean'),
        cosine_median=('cosine', 'median'),
        cosine_q25=('cosine', lambda x: x.quantile(0.25)),
        cosine_q75=('cosine', lambda x: x.quantile(0.75)),
        matched_peak_fraction_median=('matched_a_fraction', 'median'),
    ).reset_index()
)
pilot_gap_summary.to_csv(OUTPUT_DIR / 'pilot_sim_to_real_gap_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(
    data=pilot_gap_summary, x='simulated_ce_ev', y='cosine_median', hue='mode',
    marker='o', linewidth=2.2, ax=ax,
)
ax.set_ylim(0, 1)
ax.set_xlabel('CFM collision energy (eV)')
ax.set_ylabel('Median cosine to public measured spectrum')
ax.set_title('Pilot sim-to-real gap must remain visible')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '14_pilot_sim_to_real_energy_gap.png', dpi=240, bbox_inches='tight')
plt.show()
display(pilot_gap_summary)

## 3. 读取候选化合物并隔离ChemVista Novel

`compound_split_manifest.csv`是已经冻结的化合物级拆分。代码先按角色隔离External Novel，Novel侧只保留身份和结构用于
精确/近邻排除；不会统计或导出Novel的PBT标签。候选侧只保留Development与property-without-public-spectrum。

In [ ]:
PRIMARY_LABELS = {
    'P': 'label__P_screen_not_ready_binary',
    'B': 'label__B_LOGBCF_REGRESSION',
    'EC50': 'label__T_EC50_MG_L_REGRESSION',
    'LC50': 'label__T_LC50_MG_L_REGRESSION',
    'NOEC': 'label__T_NOEC_MG_L_REGRESSION',
}
identity_columns = [
    'inchikey_2d', 'canonical_smiles', 'preferred_names', 'smiles_resolution',
    'public_spectrum_count', 'has_public_spectrum', 'supervised_compound_role',
    'chemvista_evaluation_role', 'cv_group', 'cv_fold',
]
split = pd.read_csv(PATHS['compound_split'], usecols=identity_columns + list(PRIMARY_LABELS.values()))
novel = split.loc[
    split['supervised_compound_role'].astype(str).eq('external_novel'),
    identity_columns,
].copy()
candidates = split.loc[
    split['supervised_compound_role'].astype(str).isin(['development', 'property_without_public_spectrum'])
].copy()

# External Novel标签不进入任何候选对象、统计或输出。
assert len(set(candidates['inchikey_2d'].astype(str)) & set(novel['inchikey_2d'].astype(str))) == 0
assert len(novel) == split['supervised_compound_role'].astype(str).eq('external_novel').sum()
for short, column in PRIMARY_LABELS.items():
    candidates[f'has_{short}'] = pd.to_numeric(candidates[column], errors='coerce').notna()
candidates['n_primary_labels'] = candidates[[f'has_{x}' for x in PRIMARY_LABELS]].sum(axis=1).astype(int)
candidates['endpoint_signature'] = candidates.apply(
    lambda row: ''.join(short for short in PRIMARY_LABELS if bool(row[f'has_{short}'])) or 'none', axis=1
)

print('Frozen split compounds:', len(split))
print('Candidate identities before structure QC:', len(candidates))
print('ChemVista External Novel identities isolated:', len(novel))
print(candidates['supervised_compound_role'].value_counts().to_string())

## 4. CFM结构标准化和适用域

13号唯一失败是预带双负电荷的单一结构。这里允许可追踪的去电荷转换，但必须满足：

- 原始结构只有一个共价组分；
- 去电荷后没有残余形式电荷；
- 去电荷前后二维InChIKey一致；
- 元素和分子大小在冻结范围内。

含点号的盐/混合物不会自动“猜主成分”；永久带电或连接身份改变的结构也不会强行生成。

In [ ]:
uncharger = rdMolStandardize.Uncharger()
morgan = rdFingerprintGenerator.GetMorganGenerator(
    radius=MORGAN_RADIUS, fpSize=MORGAN_BITS, includeChirality=MORGAN_USE_CHIRALITY
)

def structure_record(smiles, expected_ik2d):
    result = {
        'structure_valid': False, 'cfm_smiles': '', 'cfm_ik2d': '',
        'structure_transform': 'none', 'fragment_count': np.nan,
        'original_formal_charge': np.nan, 'cfm_formal_charge': np.nan,
        'exact_mw': np.nan, 'heavy_atoms': np.nan, 'elements': '',
        'structure_exclusion_reason': '', 'fingerprint': None,
    }
    if pd.isna(smiles) or not str(smiles).strip():
        result['structure_exclusion_reason'] = 'missing_smiles'
        return result
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        result['structure_exclusion_reason'] = 'invalid_smiles'
        return result
    fragments = Chem.GetMolFrags(mol)
    result['fragment_count'] = len(fragments)
    result['original_formal_charge'] = int(Chem.GetFormalCharge(mol))
    if len(fragments) != 1:
        result['structure_exclusion_reason'] = 'salt_or_mixture_multiple_fragments'
        return result
    try:
        neutral = uncharger.uncharge(Chem.Mol(mol))
        Chem.SanitizeMol(neutral)
    except Exception:
        result['structure_exclusion_reason'] = 'uncharging_or_sanitization_failed'
        return result
    result['cfm_formal_charge'] = int(Chem.GetFormalCharge(neutral))
    if result['cfm_formal_charge'] != 0 or any(atom.GetFormalCharge() != 0 for atom in neutral.GetAtoms()):
        result['structure_exclusion_reason'] = 'residual_or_permanent_formal_charge'
        return result
    cfm_smiles = Chem.MolToSmiles(neutral, canonical=True, isomericSmiles=False)
    cfm_ik2d = Chem.MolToInchiKey(neutral).split('-')[0]
    result['cfm_smiles'] = cfm_smiles
    result['cfm_ik2d'] = cfm_ik2d
    if cfm_ik2d != str(expected_ik2d):
        result['structure_exclusion_reason'] = 'neutralization_changed_connectivity_identity'
        return result
    elements = sorted({atom.GetSymbol() for atom in neutral.GetAtoms()})
    unsupported = set(elements) - ALLOWED_ELEMENTS
    result['elements'] = ','.join(elements)
    if unsupported:
        result['structure_exclusion_reason'] = 'unsupported_elements:' + ','.join(sorted(unsupported))
        return result
    result['exact_mw'] = float(Descriptors.ExactMolWt(neutral))
    result['heavy_atoms'] = int(neutral.GetNumHeavyAtoms())
    if not (MIN_EXACT_MW <= result['exact_mw'] <= MAX_EXACT_MW):
        result['structure_exclusion_reason'] = 'exact_mw_out_of_frozen_range'
        return result
    if not (MIN_HEAVY_ATOMS <= result['heavy_atoms'] <= MAX_HEAVY_ATOMS):
        result['structure_exclusion_reason'] = 'heavy_atom_count_out_of_frozen_range'
        return result
    canonical_original = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
    result['structure_transform'] = 'uncharged_same_connectivity' if cfm_smiles != canonical_original else 'canonical_identity'
    result['structure_valid'] = True
    result['fingerprint'] = morgan.GetFingerprint(neutral)
    return result

structure_rows = []
fingerprints = {}
for row in tqdm(candidates[['inchikey_2d', 'canonical_smiles']].itertuples(index=False),
                total=len(candidates), desc='CFM structure QC'):
    record = structure_record(row.canonical_smiles, row.inchikey_2d)
    fingerprint = record.pop('fingerprint')
    record['inchikey_2d'] = row.inchikey_2d
    structure_rows.append(record)
    if fingerprint is not None:
        fingerprints[str(row.inchikey_2d)] = fingerprint
structure_qc = pd.DataFrame(structure_rows)
candidates = candidates.merge(structure_qc, on='inchikey_2d', how='left', validate='one_to_one')

print('Structurally eligible before label/Novel rules:', int(candidates['structure_valid'].sum()))
display(candidates.loc[~candidates['structure_valid'], 'structure_exclusion_reason'].value_counts().rename('n'))
display(candidates.loc[candidates['structure_transform'].eq('uncharged_same_connectivity'),
                       ['inchikey_2d', 'canonical_smiles', 'cfm_smiles', 'original_formal_charge']].head(10))

## 5. ChemVista Novel精确与近邻排除

Novel结构只用于计算排除距离。候选与Novel的最大Morgan Tanimoto达到0.80即从CFM生成计划排除。
这不是在ChemVista结果上调参，而是事先冻结的严格未知结构保护。

In [ ]:
novel_fingerprints = []
novel_keys_for_fp = []
for row in novel[['inchikey_2d', 'canonical_smiles']].itertuples(index=False):
    mol = Chem.MolFromSmiles(str(row.canonical_smiles)) if pd.notna(row.canonical_smiles) else None
    if mol is not None:
        novel_fingerprints.append(morgan.GetFingerprint(mol))
        novel_keys_for_fp.append(str(row.inchikey_2d))
assert novel_fingerprints, '没有可用于Novel近邻排除的有效结构。'

max_novel_similarity = []
nearest_novel_key = []
for key in tqdm(candidates['inchikey_2d'].astype(str), desc='Novel neighbor screen'):
    fp = fingerprints.get(key)
    if fp is None:
        max_novel_similarity.append(np.nan)
        nearest_novel_key.append('')
        continue
    similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, novel_fingerprints), dtype=float)
    best = int(np.argmax(similarities))
    max_novel_similarity.append(float(similarities[best]))
    nearest_novel_key.append(novel_keys_for_fp[best])
candidates['max_tanimoto_to_chemvista_novel'] = max_novel_similarity
candidates['nearest_chemvista_novel_ik2d'] = nearest_novel_key
candidates['chemvista_novel_neighbor_excluded'] = (
    candidates['max_tanimoto_to_chemvista_novel'].ge(NOVEL_NEIGHBOR_TANIMOTO)
)
candidates['chemvista_novel_exact_overlap'] = candidates['inchikey_2d'].astype(str).isin(
    set(novel['inchikey_2d'].astype(str))
)
assert not candidates['chemvista_novel_exact_overlap'].any()

novel_neighbors = candidates.loc[candidates['chemvista_novel_neighbor_excluded']].copy()
novel_neighbors.to_csv(
    OUTPUT_DIR / 'cfm_excluded_chemvista_novel_neighbors.csv', index=False, encoding='utf-8-sig'
)
print('Novel exact overlaps:', int(candidates['chemvista_novel_exact_overlap'].sum()))
print(f'Novel neighbors excluded at >= {NOVEL_NEIGHBOR_TANIMOTO:.2f}:', len(novel_neighbors))

## 6. 冻结候选角色

- `same_compound_condition_augmentation`：已有Development合格公共实测谱，用于同化合物不同能量/模式增强。
- `synthetic_only_new_labeled_compound`：有目标标签但没有Development合格公共谱，用于扩大标签化学空间。

两类CFM谱都只能进入训练侧；任何CFM谱都不参与内部外折测试指标或外部测试指标。

In [ ]:
public = pd.read_csv(PATHS['public_manifest'])
public_eligible = public.loc[
    public['primary_train_eligible_20ppm'].fillna(False).astype(bool)
    & public['exact_dedup_keep'].fillna(False).astype(bool)
    & public['supervised_compound_role'].astype(str).eq('development')
].copy()
development_public_keys = set(public_eligible['inchikey_2d'].astype(str))
candidates['has_development_eligible_public_spectrum'] = candidates['inchikey_2d'].astype(str).isin(development_public_keys)
candidates['generation_role'] = np.where(
    candidates['has_development_eligible_public_spectrum'],
    'same_compound_condition_augmentation',
    'synthetic_only_new_labeled_compound',
)

candidates['preliminary_include'] = (
    candidates['structure_valid']
    & candidates['n_primary_labels'].gt(0)
    & ~candidates['chemvista_novel_neighbor_excluded']
    & ~candidates['chemvista_novel_exact_overlap']
)
candidates['exclusion_reason'] = ''
candidates.loc[~candidates['structure_valid'], 'exclusion_reason'] = candidates.loc[
    ~candidates['structure_valid'], 'structure_exclusion_reason'
]
candidates.loc[candidates['structure_valid'] & candidates['n_primary_labels'].eq(0), 'exclusion_reason'] = 'no_primary_PBT_model_label'
candidates.loc[
    candidates['structure_valid'] & candidates['n_primary_labels'].gt(0)
    & candidates['chemvista_novel_neighbor_excluded'], 'exclusion_reason'
] = 'chemvista_novel_neighbor_ge_frozen_threshold'

preliminary = candidates.loc[candidates['preliminary_include']].copy()
print('Preliminary included compounds:', len(preliminary))
print(preliminary['generation_role'].value_counts().to_string())

## 7. 外折无泄漏允许表

Development化合物直接继承冻结`cv_fold`：自身测试折禁止使用CFM谱，其余折允许。

Synthetic-only化合物从不作为测试样本。对每个外折，计算它与该折全部Development测试结构的最大Tanimoto；
若达到0.40，则该化合物CFM谱在该外折训练中禁用。这样不会把新增合成结构随机放入测试侧，也不会让其作为
测试结构簇的近邻进入训练。

In [ ]:
development = pd.read_csv(PATHS['development'])
development_fps_by_fold = {}
for fold in range(5):
    keys = development.loc[pd.to_numeric(development['cv_fold'], errors='coerce').eq(fold), 'inchikey_2d'].astype(str)
    development_fps_by_fold[fold] = [fingerprints[key] for key in keys if key in fingerprints]
    assert development_fps_by_fold[fold], f'冻结外折{fold}没有有效结构指纹。'

permission_rows = []
for row in tqdm(preliminary.itertuples(index=False), total=len(preliminary), desc='Outer-fold permissions'):
    fp = fingerprints[str(row.inchikey_2d)]
    for outer_fold in range(5):
        fold_similarities = DataStructs.BulkTanimotoSimilarity(fp, development_fps_by_fold[outer_fold])
        max_similarity = float(max(fold_similarities)) if fold_similarities else np.nan
        if row.supervised_compound_role == 'development':
            allowed = int(row.cv_fold) != outer_fold
            reason = 'blocked_same_development_test_fold' if not allowed else 'allowed_development_training_side'
        else:
            allowed = bool(max_similarity < OUTER_FOLD_BLOCK_TANIMOTO)
            reason = 'allowed_synthetic_only_distant_from_test_fold' if allowed else 'blocked_neighbor_of_test_fold'
        permission_rows.append({
            'inchikey_2d': row.inchikey_2d,
            'generation_role': row.generation_role,
            'outer_fold': outer_fold,
            'allowed_in_outer_fold_training': allowed,
            'max_tanimoto_to_outer_test_fold': max_similarity,
            'permission_reason': reason,
        })
permissions = pd.DataFrame(permission_rows)
permission_counts = permissions.groupby('inchikey_2d')['allowed_in_outer_fold_training'].sum().rename('n_allowed_outer_folds')
candidates = candidates.merge(permission_counts, on='inchikey_2d', how='left')
candidates['n_allowed_outer_folds'] = candidates['n_allowed_outer_folds'].fillna(0).astype(int)
candidates['final_include'] = candidates['preliminary_include'] & candidates['n_allowed_outer_folds'].gt(0)
candidates.loc[
    candidates['preliminary_include'] & candidates['n_allowed_outer_folds'].eq(0), 'exclusion_reason'
] = 'blocked_in_all_outer_folds_by_structure_neighbors'

included_keys = set(candidates.loc[candidates['final_include'], 'inchikey_2d'].astype(str))
permissions = permissions.loc[permissions['inchikey_2d'].astype(str).isin(included_keys)].copy()
permissions.to_csv(OUTPUT_DIR / 'cfm_outer_fold_permission.csv', index=False, encoding='utf-8-sig')
print('Final included compounds:', len(included_keys))
print('Permission rows:', len(permissions))
display(permissions.groupby(['generation_role', 'outer_fold'])['allowed_in_outer_fold_training'].agg(['count', 'sum', 'mean']))

## 8. 冻结化合物表、排除表和CFM任务表

每个最终纳入化合物生成正、负两个命令；每个命令输出10/20/40 eV三张谱。CFM命令与13号完全一致，
继续使用同一个Docker镜像digest。任务表只负责映射，不在14号执行。

In [ ]:
candidates = candidates.sort_values('inchikey_2d').reset_index(drop=True)
included = candidates.loc[candidates['final_include']].copy().reset_index(drop=True)
included.insert(0, 'cfm_compound_id', [f'CFM_{i:05d}' for i in range(1, len(included) + 1)])
id_map = included.set_index('inchikey_2d')['cfm_compound_id']
candidates['cfm_compound_id'] = candidates['inchikey_2d'].map(id_map).fillna('')

compound_columns = [
    'cfm_compound_id', 'inchikey_2d', 'preferred_names', 'canonical_smiles', 'cfm_smiles',
    'structure_transform', 'original_formal_charge', 'cfm_formal_charge', 'exact_mw',
    'heavy_atoms', 'elements', 'endpoint_signature', 'n_primary_labels',
    'supervised_compound_role', 'generation_role', 'has_development_eligible_public_spectrum',
    'chemvista_evaluation_role', 'cv_group', 'cv_fold',
    'max_tanimoto_to_chemvista_novel', 'nearest_chemvista_novel_ik2d',
    'n_allowed_outer_folds', 'final_include', 'exclusion_reason',
    *PRIMARY_LABELS.values(),
]
candidates[compound_columns].to_csv(OUTPUT_DIR / 'cfm_compound_manifest.csv', index=False, encoding='utf-8-sig')
candidates.loc[~candidates['final_include'], compound_columns].to_csv(
    OUTPUT_DIR / 'cfm_structure_and_policy_exclusions.csv', index=False, encoding='utf-8-sig'
)

pilot_image = pilot_manifest['cfm']['docker_image']
pilot_digest = pilot_manifest['cfm']['docker_image_digest']
jobs = []
for row in included.itertuples(index=False):
    for mode, adduct in MODES.items():
        sign = '[M+H]+' if mode == 'positive' else '[M-H]-'
        jobs.append({
            'job_id': f'{row.cfm_compound_id}_{mode}',
            'cfm_compound_id': row.cfm_compound_id,
            'inchikey_2d': row.inchikey_2d,
            'cfm_smiles': row.cfm_smiles,
            'structure_transform': row.structure_transform,
            'generation_role': row.generation_role,
            'mode': mode,
            'adduct': adduct,
            'collision_energies_ev': '10;20;40',
            'probability_threshold': pilot_manifest['cfm']['probability_threshold'],
            'model_param': f'/trained_models_cfmid4.0/{sign}/param_output.log',
            'model_config': f'/trained_models_cfmid4.0/{sign}/param_config.txt',
            'docker_image': pilot_image,
            'docker_image_digest': pilot_digest,
            'output_relative_path': f'raw/{mode}/{row.cfm_compound_id}_{row.inchikey_2d}.txt',
        })
jobs = pd.DataFrame(jobs)
assert len(jobs) == len(included) * 2
assert not jobs.duplicated(['inchikey_2d', 'mode']).any()
jobs.to_csv(OUTPUT_DIR / 'cfm_generation_jobs.csv', index=False, encoding='utf-8-sig')

print('Included compounds:', len(included))
print('CFM command jobs:', len(jobs))
print('Expected spectra:', len(jobs) * 3)
display(included['generation_role'].value_counts().rename('n_compounds'))
display(candidates.loc[~candidates['final_include'], 'exclusion_reason'].value_counts().rename('n_excluded'))

## 9. 冻结15号实测对照配对表

只使用公共Development实测谱。对每个纳入且有实测谱的化合物、模式和10/20/40 eV，预先选择同加合物且
碰撞能量差最小的实测谱；差值超过10 eV的任务标为没有严格对照，不强行填零。

该表只定义“将来比较谁和谁”，真正的谱峰差距指标由15号计算。

In [ ]:
public_pair_pool = public_eligible.loc[
    public_eligible['inchikey_2d'].astype(str).isin(included_keys)
    & pd.to_numeric(public_eligible['collision_energy_value'], errors='coerce').notna()
].copy()
public_pair_pool['mode'] = public_pair_pool['ion_mode'].astype(str).str.lower()
public_pair_pool = public_pair_pool.loc[
    public_pair_pool.apply(lambda row: str(row['adduct']) == MODES.get(row['mode']), axis=1)
].copy()
public_pair_pool['collision_energy_value'] = pd.to_numeric(public_pair_pool['collision_energy_value'], errors='coerce')

reference_rows = []
for job in tqdm(jobs.itertuples(index=False), total=len(jobs), desc='Freeze real-spectrum references'):
    options = public_pair_pool.loc[
        public_pair_pool['inchikey_2d'].astype(str).eq(str(job.inchikey_2d))
        & public_pair_pool['mode'].eq(job.mode)
    ].copy()
    for energy in COLLISION_ENERGIES_EV:
        if len(options):
            ranked = options.assign(ce_delta=(options['collision_energy_value'] - energy).abs()).sort_values(
                ['ce_delta', 'cleaned_peak_n', 'identity_ppm_error', 'spectrum_id'],
                ascending=[True, False, True, True],
            )
            best = ranked.iloc[0]
            strict_pair = bool(float(best['ce_delta']) <= 10.0)
            spectrum_id = str(best['spectrum_id']) if strict_pair else ''
            measured_ce = float(best['collision_energy_value']) if strict_pair else np.nan
            ce_delta = float(best['ce_delta']) if strict_pair else np.nan
            stream_offset = int(best['stream_offset']) if strict_pair else np.nan
            peak_hash = str(best['peak_hash_common']) if strict_pair else ''
        else:
            strict_pair = False; spectrum_id = ''; measured_ce = np.nan
            ce_delta = np.nan; stream_offset = np.nan; peak_hash = ''
        reference_rows.append({
            'job_id': job.job_id,
            'cfm_compound_id': job.cfm_compound_id,
            'inchikey_2d': job.inchikey_2d,
            'mode': job.mode,
            'adduct': job.adduct,
            'simulated_ce_ev': energy,
            'strict_public_reference_available': strict_pair,
            'measured_spectrum_id': spectrum_id,
            'measured_ce_ev': measured_ce,
            'ce_delta_ev': ce_delta,
            'measured_stream_offset': stream_offset,
            'measured_peak_hash': peak_hash,
        })
references = pd.DataFrame(reference_rows)
references.to_csv(OUTPUT_DIR / 'cfm_sim_to_real_reference_pairs.csv', index=False, encoding='utf-8-sig')
print('Expected synthetic spectra:', len(references))
print('Strict public real-spectrum references:', int(references['strict_public_reference_available'].sum()))
display(references.groupby(['mode', 'simulated_ce_ev'])['strict_public_reference_available'].agg(['count', 'sum', 'mean']))

## 10. 候选流、标签覆盖和外折可用性图

In [ ]:
flow = pd.DataFrame([
    {'stage': 'Candidate identities', 'n': len(candidates)},
    {'stage': 'Valid CFM structure', 'n': int(candidates['structure_valid'].sum())},
    {'stage': 'At least one primary label', 'n': int((candidates['structure_valid'] & candidates['n_primary_labels'].gt(0)).sum())},
    {'stage': 'After Novel-neighbor exclusion', 'n': int(candidates['preliminary_include'].sum())},
    {'stage': 'At least one allowed outer fold', 'n': int(candidates['final_include'].sum())},
])
flow.to_csv(OUTPUT_DIR / 'cfm_candidate_flow.csv', index=False)

endpoint_coverage = []
for role, group in included.groupby('generation_role'):
    for endpoint in PRIMARY_LABELS:
        endpoint_coverage.append({
            'generation_role': role, 'endpoint': endpoint,
            'n_labeled_compounds': int(group[f'has_{endpoint}'].sum()),
        })
endpoint_coverage = pd.DataFrame(endpoint_coverage)
endpoint_coverage.to_csv(OUTPUT_DIR / 'cfm_endpoint_coverage.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
sns.barplot(data=flow, x='n', y='stage', color='#4472C4', ax=axes[0])
axes[0].set_title('CFM candidate flow')
axes[0].set_xlabel('Compounds')
sim_values = candidates.loc[candidates['structure_valid'], 'max_tanimoto_to_chemvista_novel'].dropna()
sns.histplot(sim_values, bins=30, ax=axes[1], color='#70AD47')
axes[1].axvline(NOVEL_NEIGHBOR_TANIMOTO, color='red', linestyle='--', label='Frozen exclusion')
axes[1].set_title('Similarity to ChemVista Novel')
axes[1].legend()
permission_plot = permissions.groupby(['generation_role', 'outer_fold'])['allowed_in_outer_fold_training'].sum().reset_index(name='n_allowed')
sns.barplot(data=permission_plot, x='outer_fold', y='n_allowed', hue='generation_role', ax=axes[2])
axes[2].set_title('Permitted CFM compounds per outer fold')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '14_generation_plan_overview.png', dpi=240, bbox_inches='tight')
plt.show()
display(flow)
display(endpoint_coverage)

## 11. 硬性QC与正式冻结

14号通过后，15号必须严格读取这里的三个核心表：`cfm_generation_jobs.csv`、
`cfm_outer_fold_permission.csv`和`cfm_sim_to_real_reference_pairs.csv`。不得在15号中临时新增结构或修改Novel阈值。

In [ ]:
included_manifest = candidates.loc[candidates['final_include']].copy()
development_included = included_manifest['supervised_compound_role'].astype(str).eq('development')
development_permission = permissions.loc[
    permissions['inchikey_2d'].astype(str).isin(set(included_manifest.loc[development_included, 'inchikey_2d'].astype(str)))
].merge(included_manifest[['inchikey_2d', 'cv_fold']], on='inchikey_2d', how='left')
own_fold_leaks = development_permission.loc[
    development_permission['outer_fold'].eq(pd.to_numeric(development_permission['cv_fold']))
    & development_permission['allowed_in_outer_fold_training']
]

permission_n_by_key = permissions.groupby('inchikey_2d').size()
job_modes_by_key = jobs.groupby('inchikey_2d')['mode'].agg(lambda x: set(x))
exact_novel_overlap = len(included_keys & set(novel['inchikey_2d'].astype(str)))
near_novel_included = int(included_manifest['max_tanimoto_to_chemvista_novel'].ge(NOVEL_NEIGHBOR_TANIMOTO).sum())
invalid_structure_included = int((~included_manifest['structure_valid']).sum())
no_label_included = int(included_manifest['n_primary_labels'].eq(0).sum())

qc = pd.DataFrame([
    {'check': '13_pilot_accepted', 'passed': PILOT_ACCEPTED, 'observed': PILOT_DECISION, 'expected': 'accepted'},
    {'check': 'candidate_identity_unique', 'passed': candidates['inchikey_2d'].is_unique, 'observed': candidates['inchikey_2d'].nunique(), 'expected': len(candidates)},
    {'check': 'external_novel_exact_overlap_zero', 'passed': exact_novel_overlap == 0, 'observed': exact_novel_overlap, 'expected': 0},
    {'check': 'novel_neighbors_included_zero', 'passed': near_novel_included == 0, 'observed': near_novel_included, 'expected': 0},
    {'check': 'invalid_structures_included_zero', 'passed': invalid_structure_included == 0, 'observed': invalid_structure_included, 'expected': 0},
    {'check': 'included_without_primary_label_zero', 'passed': no_label_included == 0, 'observed': no_label_included, 'expected': 0},
    {'check': 'every_included_has_five_permission_rows', 'passed': permission_n_by_key.eq(5).all() and len(permission_n_by_key) == len(included), 'observed': len(permission_n_by_key), 'expected': len(included)},
    {'check': 'development_own_fold_training_leaks_zero', 'passed': len(own_fold_leaks) == 0, 'observed': len(own_fold_leaks), 'expected': 0},
    {'check': 'every_included_has_positive_and_negative_jobs', 'passed': job_modes_by_key.map(lambda x: x == set(MODES)).all() and len(job_modes_by_key) == len(included), 'observed': len(job_modes_by_key), 'expected': len(included)},
    {'check': 'job_count_equals_compounds_times_two', 'passed': len(jobs) == len(included) * 2, 'observed': len(jobs), 'expected': len(included) * 2},
    {'check': 'reference_rows_equals_jobs_times_three', 'passed': len(references) == len(jobs) * 3, 'observed': len(references), 'expected': len(jobs) * 3},
    {'check': 'all_included_have_at_least_one_allowed_fold', 'passed': included_manifest['n_allowed_outer_folds'].gt(0).all(), 'observed': int(included_manifest['n_allowed_outer_folds'].gt(0).sum()), 'expected': len(included_manifest)},
])
qc.to_csv(OUTPUT_DIR / 'cfm_generation_plan_qc.csv', index=False, encoding='utf-8-sig')
PLAN_STATUS = 'READY_FOR_NOTEBOOK_15' if qc['passed'].all() else 'BLOCKED_REVIEW_QC'
display(qc)
print('PLAN_STATUS:', PLAN_STATUS)
assert qc['passed'].all(), '14号存在未通过的硬性QC，不得进入15号。'

In [ ]:
formal_outputs = [
    OUTPUT_DIR / '13_pilot_acceptance_for_14.csv',
    OUTPUT_DIR / 'pilot_sim_to_real_gap_summary.csv',
    OUTPUT_DIR / 'cfm_compound_manifest.csv',
    OUTPUT_DIR / 'cfm_structure_and_policy_exclusions.csv',
    OUTPUT_DIR / 'cfm_excluded_chemvista_novel_neighbors.csv',
    OUTPUT_DIR / 'cfm_outer_fold_permission.csv',
    OUTPUT_DIR / 'cfm_generation_jobs.csv',
    OUTPUT_DIR / 'cfm_sim_to_real_reference_pairs.csv',
    OUTPUT_DIR / 'cfm_candidate_flow.csv',
    OUTPUT_DIR / 'cfm_endpoint_coverage.csv',
    OUTPUT_DIR / 'cfm_generation_plan_qc.csv',
    FIGURE_DIR / '14_pilot_sim_to_real_energy_gap.png',
    FIGURE_DIR / '14_generation_plan_overview.png',
]
assert all(path.exists() for path in formal_outputs)

manifest = {
    'notebook': '14_CFM扩增化合物冻结与无泄漏折分配.ipynb',
    'generated_at_utc': utc_now(),
    'status': PLAN_STATUS,
    'scientific_scope': 'Development and property-without-public-spectrum CFM generation plan; ChemVista remains locked',
    'external_test_unlocked': False,
    'pilot_decision': PILOT_DECISION,
    'structure_policy': {
        'allowed_elements': sorted(ALLOWED_ELEMENTS),
        'exact_mw_range': [MIN_EXACT_MW, MAX_EXACT_MW],
        'heavy_atom_range': [MIN_HEAVY_ATOMS, MAX_HEAVY_ATOMS],
        'multiple_fragments': 'exclude',
        'charge_policy': 'uncharger allowed only if IK2D is unchanged and no formal charge remains',
    },
    'leakage_policy': {
        'morgan_radius': MORGAN_RADIUS, 'morgan_bits': MORGAN_BITS,
        'morgan_use_chirality': MORGAN_USE_CHIRALITY,
        'chemvista_novel_neighbor_exclusion_tanimoto': NOVEL_NEIGHBOR_TANIMOTO,
        'synthetic_outer_fold_block_tanimoto': OUTER_FOLD_BLOCK_TANIMOTO,
        'development_compound_rule': 'blocked in its own frozen test fold',
        'synthetic_only_rule': 'training-only and blocked when near an outer test fold',
    },
    'cfm': {
        'docker_image': pilot_manifest['cfm']['docker_image'],
        'docker_image_digest': pilot_manifest['cfm']['docker_image_digest'],
        'probability_threshold': pilot_manifest['cfm']['probability_threshold'],
        'modes': MODES, 'collision_energies_ev': COLLISION_ENERGIES_EV,
    },
    'counts': {
        'candidate_identities': len(candidates),
        'included_compounds': len(included),
        'excluded_compounds': int((~candidates['final_include']).sum()),
        'same_compound_augmentation': int(included['generation_role'].eq('same_compound_condition_augmentation').sum()),
        'synthetic_only_new_labeled': int(included['generation_role'].eq('synthetic_only_new_labeled_compound').sum()),
        'generation_command_jobs': len(jobs),
        'expected_simulated_spectra': len(jobs) * 3,
        'strict_public_reference_pairs': int(references['strict_public_reference_available'].sum()),
        'novel_neighbors_excluded': len(novel_neighbors),
    },
    'sim_to_real_plan': {
        'pilot_pairs': len(pilot_sim_to_real),
        'full_audit_notebook': '15_CFM批量谱图整合与Sim_to_Real审计.ipynb',
        'frozen_metrics': ['modified_cosine', 'spectral_entropy_similarity', 'peak_recall',
                           'intensity_correlation', 'embedding_distance', 'domain_classifier'],
        'selection_use': 'CFM training role/weight only; never tune from ChemVista outcomes',
    },
    'input_hashes': {name: sha256_file(path) for name, path in PATHS.items()},
    'output_hashes': {str(path.relative_to(OUTPUT_DIR)): sha256_file(path) for path in formal_outputs},
    'software': {'python': sys.version, 'platform': platform.platform(), 'rdkit': rdBase.rdkitVersion},
}
temporary_manifest = FROZEN_MANIFEST_PATH.with_suffix('.json.tmp')
temporary_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
os.replace(temporary_manifest, FROZEN_MANIFEST_PATH)
print('14号冻结完成：', PLAN_STATUS)
print('输出目录：', OUTPUT_DIR)
print('下一步：15号按 cfm_generation_jobs.csv 批量生成并完成正式sim-to-real审计。')

## 运行结束后的解释

14号通过只表示“结构与任务计划无泄漏且可执行”，不表示CFM谱已经足够接近真实谱。

15号必须首先生成全部任务，然后在冻结实测对照上评估域差距。根据结果采取以下预先约定决策：

- 域差距较小且分层稳定：可测试同化合物条件增强，但仍保留synthetic标记；
- 中等差距或高能量明显变差：优先用于自监督预训练、低权重一致性学习或仅使用10/20 eV；
- real-vs-synthetic域分类器几乎完美区分：CFM不得作为与实测谱同权重的监督样本；
- 所有决策只根据Development公共实测谱制定，不能查看ChemVista结果后再修改。